In [2]:
import pandas as pd

textes = [
    # positifs
    "J'adore ce film, il est magnifique",
    "Service excellent, je suis très content",
    "Ce produit est génial et utile",
    "Une expérience formidable, je recommande",
    "Le repas était délicieux",
    "Je suis ravi de mon achat",
    "Super application, simple et rapide",
    "Un livre passionnant, je l'ai adoré",
    "Très bonne qualité, je suis satisfait",
    "Un moment merveilleux, quelle belle soirée",
    # négatifs
    "Je déteste ce film, il est ennuyeux",
    "Service horrible, je suis très déçu",
    "Ce produit est nul et inutile",
    "Une expérience terrible, je ne recommande pas",
    "Le repas était froid et mauvais",
    "Je regrette mon achat",
    "Application lente et pleine de bugs",
    "Un livre ennuyeux, je n'ai pas aimé",
    "Mauvaise qualité, je suis mécontent",
    "Un moment pénible, quelle perte de temps",
]

labels = [1] * 10 + [0] * 10   # 1 = positif, 0 = négatif

df = pd.DataFrame({"texte": textes, "label": labels})

print(df.shape)
print(df["label"].value_counts())
df.head()

(20, 2)
label
1    10
0    10
Name: count, dtype: int64


,texte,label
0,"J'adore ce film, il est magnifique",1
1,"Service excellent, je suis très content",1
2,Ce produit est génial et utile,1
3,"Une expérience formidable, je recommande",1
4,Le repas était délicieux,1


In [3]:
import re
import spacy
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nlp = spacy.load("fr_core_news_sm")

stop_fr = set(stopwords.words('french'))
stop_fr = stop_fr - {"ne", "pas", "n"}   # kan-khlliw la négation

def pretraiter(texte):
    texte = texte.lower()
    texte = re.sub(r"[^\w\s]", " ", texte)
    tokens = word_tokenize(texte, language='french')
    tokens = [t for t in tokens if t not in stop_fr]
    doc = nlp(" ".join(tokens))
    return " ".join([mot.lemma_ for mot in doc])

df["texte_propre"] = df["texte"].apply(pretraiter)
df[["texte", "texte_propre"]].head()

,texte,texte_propre
0,"J'adore ce film, il est magnifique",ador film magnifique
1,"Service excellent, je suis très content",service excellent très content
2,Ce produit est génial et utile,produit génial utile
3,"Une expérience formidable, je recommande",expérience formidable recommander
4,Le repas était délicieux,repa délicieux


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["texte_propre"])
y = df["label"]

print("Forme de la matrice :", X.shape)

Forme de la matrice : (20, 53)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Entraînement :", X_train.shape[0], "textes")
print("Test :", X_test.shape[0], "textes")

Entraînement : 16 textes
Test : 4 textes


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

modeles = {
    "Régression logistique": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "SVM linéaire": LinearSVC(),
}

for nom, modele in modeles.items():
    scores = cross_val_score(modele, X_train, y_train, cv=5)
    print(nom, ":", round(scores.mean(), 2))

Régression logistique : 0.25
Naive Bayes : 0.2
SVM linéaire : 0.25
